In [1]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
from google.colab import drive
drive.mount('/content/drive')
path = "/content/drive/MyDrive/NLP_2026/"

Mounted at /content/drive


In [3]:
files = [
    "reviews_0-250_masked.csv",
    "reviews_250-500_masked.csv",
    "reviews_500-750_masked.csv",
    "reviews_750-1250_masked.csv",
    "reviews_1250-end_masked.csv"
]

df_reviews = pd.concat(
    [pd.read_csv(path + f, encoding='latin-1') for f in files],
    ignore_index=True
)

print("Jumlah review:", df_reviews.shape)
df_reviews.head()

Jumlah review: (285412, 19)


,Unnamed: 0.1,Unnamed: 0,rating,is_recommended,helpfulness,total_feedback_count,total_neg_feedback_count,total_pos_feedback_count,submission_time,review_text,review_title,skin_tone,eye_color,skin_type,hair_color,product_id,product_name,brand_name,price_usd
0,0,0,5,1.0,1.0,2,0,2,2023-02-01,I use this with the Nudestix âCitrus Clean B...,Taught me how to double cleanse!,NaN,brown,dry,black,P504322,Gentle Hydra-Gel Face Cleanser,NUDESTIX,19.0
1,1,1,1,0.0,NaN,0,0,0,2023-03-21,I bought this lip mask after reading the revie...,Disappointed,NaN,NaN,NaN,NaN,P420652,Lip Sleeping Mask Intense Hydration with Vitam...,LANEIGE,24.0
2,2,2,5,1.0,NaN,0,0,0,2023-03-21,My review title says it all! I get so excited ...,New Favorite Routine,light,brown,dry,blonde,P420652,Lip Sleeping Mask Intense Hydration with Vitam...,LANEIGE,24.0
3,3,3,5,1.0,NaN,0,0,0,2023-03-20,Iâve always loved this formula for a long ti...,Can't go wrong with any of them,NaN,brown,combination,black,P420652,Lip Sleeping Mask Intense Hydration with Vitam...,LANEIGE,24.0
4,4,4,5,1.0,NaN,0,0,0,2023-03-20,"If you have dry cracked lips, this is a must h...",A must have !!!,light,hazel,combination,NaN,P420652,Lip Sleeping Mask Intense Hydration with Vitam...,LANEIGE,24.0


In [4]:
df_product = pd.read_csv(path + "product_info_skincare.csv", encoding='latin-1')

print("Jumlah produk:", df_product.shape)
df_product.head()

Jumlah produk: (1813, 28)


,Unnamed: 0,product_id,product_name,brand_id,brand_name,loves_count,rating,reviews,size,variation_type,...,online_only,out_of_stock,sephora_exclusive,highlights,primary_category,secondary_category,tertiary_category,child_count,child_max_price,child_min_price
0,18,P483068,ABBOTT Sampler Set,6485,ABBOTT,4493,4.8163,49.0,NaN,NaN,...,0,1,0,"['Vegan', 'Woody & Earthy Scent', 'Clean + Pla...",Fragrance,Value & Gift Sets,Perfume Gift Sets,0,NaN,NaN
1,47,P474806,Blue Tansy Reparative Mask,6321,adwoa beauty,14660,4.7581,492.0,16 oz/ 453 mg,Size,...,0,0,1,"['Good for: Damage', 'Good for: Color Care', '...",Hair,Hair Styling & Treatments,Hair Masks,0,NaN,NaN
2,48,P457233,Baomint Leave In Conditioning Styler,6321,adwoa beauty,13333,4.3472,144.0,14 oz/ 414 mL,Size,...,0,0,1,"['Clean at Sephora', 'All Hair Types', 'Curl-E...",Hair,Hair Styling & Treatments,Leave-In Conditioner,1,13.0,13.0
3,49,P474808,Blue Tansy Leave in Conditioning Styler,6321,adwoa beauty,11674,4.5762,210.0,14 oz/ 414 mL,Size,...,0,0,0,"['Good for: Damage', 'Vegan', 'Clean at Sephor...",Hair,Hair Styling & Treatments,Leave-In Conditioner,0,NaN,NaN
4,50,P457234,Baomint Moisturizing Shampoo,6321,adwoa beauty,11122,4.1324,136.0,14 oz/ 414 mL,Size,...,0,0,1,"['Unisex/ Genderless Scent', 'Clean at Sephora...",Hair,Shampoo & Conditioner,Shampoo,1,12.0,12.0


In [6]:
df = pd.merge(df_reviews, df_product, on='product_id', how='left')
print(df.columns)

Index(['Unnamed: 0.1', 'Unnamed: 0_x', 'rating_x', 'is_recommended',
       'helpfulness', 'total_feedback_count', 'total_neg_feedback_count',
       'total_pos_feedback_count', 'submission_time', 'review_text',
       'review_title', 'skin_tone', 'eye_color', 'skin_type', 'hair_color',
       'product_id', 'product_name_x', 'brand_name_x', 'price_usd_x',
       'Unnamed: 0_y', 'product_name_y', 'brand_id', 'brand_name_y',
       'loves_count', 'rating_y', 'reviews', 'size', 'variation_type',
       'variation_value', 'variation_desc', 'ingredients', 'price_usd_y',
       'value_price_usd', 'sale_price_usd', 'limited_edition', 'new',
       'online_only', 'out_of_stock', 'sephora_exclusive', 'highlights',
       'primary_category', 'secondary_category', 'tertiary_category',
       'child_count', 'child_max_price', 'child_min_price'],
      dtype='object')


In [8]:
df = df.rename(columns={"product_name_y": "product_name"})

df = df[["product_id", "product_name", "review_text"]]

In [9]:
print(df.columns)

Index(['product_id', 'product_name', 'review_text'], dtype='object')


In [10]:
# hapus data kosong
df = df.dropna(subset=["product_name", "review_text"])

# ubah ke string + lowercase + strip
df["review_text"] = df["review_text"].astype(str).str.lower().str.strip()

# hapus duplikasi
df = df.drop_duplicates().reset_index(drop=True)

print(df.shape)

(284844, 3)


In [11]:
df_grouped = df.groupby("product_name")["review_text"] \
               .apply(lambda x: " ".join(x)) \
               .reset_index()

print(df_grouped.shape)
df_grouped.head()

(540, 2)


,product_name,review_text
0,1 Minute Face Masks,"for what it is, a one minute face mask, it is ..."
1,10% Benzoyl Peroxide Acne Cleanser,iâm a woman in my mid 30s that recently star...
2,10% Niacinamide Night Mask,i work at sephora and received this product in...
3,15% Niacinamide Gel Serum,"seemed to help with pore size, but left starte..."
4,2-in-1 Cleansing Oil + Makeup Remover,i received a trial size bottle of this from fa...


In [12]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    stop_words='english',
    max_features=5000
)

tfidf_matrix = tfidf.fit_transform(df_grouped["review_text"])

print(tfidf_matrix.shape)

(540, 5000)


In [13]:
from sklearn.metrics.pairwise import cosine_similarity

cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

In [14]:
def rekomendasi_produk(nama_produk, top_n=5):
    idx = df_grouped[df_grouped["product_name"].str.contains(nama_produk, case=False, na=False)].index

    if len(idx) == 0:
        return "Produk tidak ditemukan"

    idx = idx[0]

    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    sim_scores = sim_scores[1:top_n+1]

    product_indices = [i[0] for i in sim_scores]
    similarity_scores = [i[1] for i in sim_scores]

    hasil = df_grouped.iloc[product_indices][["product_name"]].copy()
    hasil["similarity_score"] = similarity_scores

    return hasil

In [15]:
idx = df_grouped[df_grouped["product_name"].str.contains("product_name", case=False, na=False)].index

In [16]:
rekomendasi_produk("hydrating", top_n=5)

,product_name,similarity_score
445,Superfood Antioxidant Cleanser,0.831940
311,Mini Superfood Antioxidant Cleanser,0.831919
428,Squalane + Amino Aloe Gentle Pore-Minimizing C...,0.819194
64,Blueberry Bounce Gentle Cleanser,0.807894
52,Barrier+ Foaming Oil Hydrating Cleanser,0.801639


In [17]:
def rekomendasi_dari_input(user_input, top_n=5):
    user_vec = tfidf.transform([user_input])

    sim_scores = cosine_similarity(user_vec, tfidf_matrix).flatten()
    sim_scores = list(enumerate(sim_scores))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    top_scores = sim_scores[:top_n]

    product_indices = [i[0] for i in top_scores]
    similarity_scores = [i[1] for i in top_scores]

    hasil = df_grouped.iloc[product_indices][["product_name"]].copy()
    hasil["similarity_score"] = similarity_scores

    return hasil

In [18]:
while True:
    user_input = input("\nCari produk (ketik 'exit' untuk keluar): ")

    if user_input.lower() == "exit":
        print("Program selesai.")
        break

    hasil = rekomendasi_dari_input(user_input, top_n=5)

    print("\nRekomendasi produk:")
    print(hasil)


Cari produk (ketik 'exit' untuk keluar): oily

Rekomendasi produk:
                                          product_name  similarity_score
134  DayWear Matte Oil-Control Anti-Oxidant Moistur...          0.445779
494               Ultra Repair Oil-Control Moisturizer          0.371135
149            Dramatically Different Moisturizing Gel          0.317563
486                    Ultra Facial Oil-Free Gel Cream          0.306449
358  Priming Moisturizer Balance Oil-Control Gel-Cream          0.300233

Cari produk (ketik 'exit' untuk keluar): redness

Rekomendasi produk:
                                          product_name  similarity_score
371  Redness Recovery+ Antioxidant Redness Treatmen...          0.407705
372                Redness Solutions Soothing Cleanser          0.379662
162              Evercalm Ultra Comforting Rescue Mask          0.172399
148  Dr. Andrew Weil for Origins Mega-Mushroom Reli...          0.129572
290  Mini Floral Recovery Overnight Mask with Squalane    

bikin tampilan web pakai Streamlit (biar keliatan keren pas demo) oke - enji